## Labour market analysis for Wales
### Basic green jobs analysis
-  How many green jobs are available in Wales and what share of the job market does this represent? What characteristics do these jobs have?
- Are there differences in the proportion of green jobs in Wales compared to the rest of the UK?
- Within Wales, what is the geography of green job offers? Is there a South / North divide?
- Has there been an increase in green job offers? 
### More complicated
- Are green jobs available in Wales of the same quality as the jobs in England and Scotland? Based on what criteria?
- Can we use ads to make inferences about the growth or demise of some industries? 
- What are the in-demand green skills profiles we can observe in different parts of Wales?



In [1]:
import polars as pl

### Load data and subset for Wales

In [2]:
created_date = pl.read_parquet("s3://prinz-green-jobs/outputs/data/ojo_application/deduplicated_sample/20241114/latest_update_20241114_key_columns.parquet")

In [3]:
combined_all_data_orig = pl.read_parquet("s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/20241121/combined_green_measures_and_meta.parquet"
)

In [4]:
# Add creation date
combined_all_data = combined_all_data_orig.join(
    created_date["id", "created"],
    left_on = "job_id",
    right_on = "id",
    how="left",
).drop('__index_level_0__')

combined_all_data = combined_all_data.with_columns(
    pl.col("created").dt.date().alias("date"),
)

combined_all_data = combined_all_data.with_columns(
    (pl.col("date").dt.month_start()).alias("month"),
    (pl.col("date").dt.year()).alias("year"),
    (pl.col("date").dt.quarter()).alias("quarter")
)

combined_all_data = combined_all_data.with_columns(
    (
        pl.col('year').cast(pl.String) + " Q" + pl.col('quarter').cast(pl.String)
    ).alias("y_quarter")
)

In [5]:
combined_all_data = combined_all_data.with_columns(
    pl.when(
        pl.col('itl_1_name').is_in(['North East (England)',
 'London',
 'South West (England)',
 'East Midlands (England)',
 'West Midlands (England)',
 'East of England',
 'South East (England)',
 'Northern Ireland',
 'North West (England)',
 'Yorkshire and the Humber'])).then(pl.lit("England")).otherwise(pl.col('itl_1_name')).alias("country")
)

In [6]:
welsh_combined_all_data = combined_all_data.filter(
    pl.col("itl_1_name")=="Wales")
len(welsh_combined_all_data)

139337

In [7]:
england_combined_all_data = combined_all_data.filter(
    pl.col("itl_1_name").is_in([
 'North East (England)',
 'London',
 'South West (England)',
 'East Midlands (England)',
 'West Midlands (England)',
 'East of England',
 'South East (England)',
 'Northern Ireland',
 'North West (England)',
 'Yorkshire and the Humber',]))
print(len(england_combined_all_data))

scotland_combined_all_data = combined_all_data.filter(
    pl.col("itl_1_name")=="Scotland")
print(len(scotland_combined_all_data))

5458040
197598


In [8]:
print(f"{len(welsh_combined_all_data)*100/len(combined_all_data)}% of job adverts ({len(welsh_combined_all_data)}) are for Wales")
num_null = len(combined_all_data.filter(pl.col('itl_1_name').is_null()))
print(f"{num_null*100/len(combined_all_data)}% of job adverts don't have a ITL 1 given, in some cases these will be for Wales")

2.335036915794584% of job adverts (139337) are for Wales
2.886666491264203% of job adverts don't have a ITL 1 given, in some cases these will be for Wales


## High level numbers

In [9]:
print(len(combined_all_data))
print(len(welsh_combined_all_data))
print(len(england_combined_all_data))
print(len(scotland_combined_all_data))

5967229
139337
5458040
197598


In [15]:
def get_missind_props(data):
    a = data['SOC_2020_name'].is_null().sum()*100/len(data)
    print(f"{round(a,3)}% of job adverts have SOCs missing")
    b = data['SIC_2_digit'].is_null().sum()*100/len(data)
    print(f"{round(b,3)}% of job adverts have SICs missing")

In [16]:
print("In Wales:")
get_missind_props(welsh_combined_all_data)

In Wales:
13.381% of job adverts have SOCs missing
36.4% of job adverts have SICs missing


In [17]:
print("In England:")
get_missind_props(england_combined_all_data)

In England:
15.355% of job adverts have SOCs missing
34.438% of job adverts have SICs missing


## Investigate

Occupation aggregates and number of skills, salaries being asked for.
-  How many green jobs are available in Wales and what share of the job market does this represent? What characteristics do these jobs have?

Table with average scores over time in Wales and in the rest of the UK. AND overall averages.
- Has there been an increase in green job offers?
- Are there differences in the proportion of green jobs in Wales compared to the rest of the UK?
- How many green jobs are available in Wales and what share of the job market does this represent? What characteristics do these jobs have?

Table with average score per Welsh location
- Within Wales, what is the geography of green job offers? Is there a South / North divide?

In [18]:
def get_green_averages(df, group_by_col):
    grouped_df = (df.group_by(group_by_col, maintain_order=True)
     .agg(
         pl.col("PROP_GREEN").mean().alias("average_prop_green_skills"),
         pl.col('INDUSTRY GHG PER UNIT EMISSIONS').mean().alias("average_ghg"),
         pl.col('GREEN TIMESHARE').mean().alias("average_green_timeshare"),
         pl.col('job_id').n_unique().alias("number_job_ids"),
     ))

    return grouped_df.unpivot(
        on=[
            "average_prop_green_skills",
            "average_ghg",
            "average_green_timeshare",
            "number_job_ids"
        ],
        index=group_by_col)
    


In [19]:
def join_time_averages(
    combined_all_data,
    welsh_combined_all_data,
    england_combined_all_data,
    scotland_combined_all_data,
    group_by_col = "year"
    ):
    all_averages_per_group = get_green_averages(combined_all_data, group_by_col).rename(
        {"value": "All"})
    welsh_averages_per_group = get_green_averages(welsh_combined_all_data, group_by_col).rename(
        {"value": "Wales"})
    england_averages_per_group = get_green_averages(england_combined_all_data, group_by_col).rename(
        {"value": "England"})
    scotland_averages_per_group = get_green_averages(scotland_combined_all_data, group_by_col).rename(
        {"value": "Scotland"})

    joined_grouped_averages = all_averages_per_group.join(
        welsh_averages_per_group, on= [group_by_col, "variable"])
    joined_grouped_averages = joined_grouped_averages.join(
        england_averages_per_group, on= [group_by_col, "variable"])
    joined_grouped_averages = joined_grouped_averages.join(
        scotland_averages_per_group, on= [group_by_col, "variable"])

    return joined_grouped_averages
    


In [20]:
joined_yearly_averages = join_time_averages(
    combined_all_data,
    welsh_combined_all_data,
    england_combined_all_data,
    scotland_combined_all_data,
    group_by_col = "year"
    )
joined_yearly_averages.write_csv("joined_yearly_averages.csv")

In [139]:
joined_monthly_averages = join_time_averages(
    combined_all_data,
    welsh_combined_all_data,
    england_combined_all_data,
    scotland_combined_all_data,
    group_by_col = "month"
    )
# Don't include lastest month in output since it was an incomplete month for data collection
joined_monthly_averages.filter(pl.col("month")<pl.date(2024, 11, 1)
                              ).write_csv("joined_monthly_averages.csv")

In [21]:
joined_quarterly_averages = join_time_averages(
    combined_all_data,
    welsh_combined_all_data,
    england_combined_all_data,
    scotland_combined_all_data,
    group_by_col = "y_quarter"
    )
# Don't include first or last quarter output since they were incomplete months for data collection
# Don't include Q4 2022 or Q1 2023 since there was a problem collecting data in these months so the numbers are low
joined_quarterly_averages.filter(~pl.col("y_quarter").is_in(["2020 Q4", "2024 Q4", "2022 Q4", "2023 Q1"])
                              ).write_csv("joined_quarter_averages.csv")

### 2024 comparisons

In [154]:
joined_yearly_averages.filter(pl.col('year')==2024).rename(
    {"variable": "variable_x"}).unpivot(
    on=["All", "Wales", "England", "Scotland"],
    index = ["year", "variable_x"]
).rename({"variable_x": "variable", "variable": "Country"}).write_csv("2024_averages.csv")

## Proportion of green jobs
How many green jobs are available in Wales and what share of the job market does this represent? What characteristics do these jobs have?

In [185]:
grouped_df = (welsh_combined_all_data.group_by("y_quarter", maintain_order=True)
     .agg(
         (pl.col("GREEN/NOT GREEN")=="Non-green").sum().alias("Number non-green"),
         (pl.col("GREEN/NOT GREEN")=="Green").sum().alias("Number green"),
         (pl.col("GREEN/NOT GREEN").is_null()).sum().alias("Number with no SOC"),
         pl.col("GREEN/NOT GREEN").len().alias("Total number"),
     ))

grouped_df = grouped_df.with_columns(
    (pl.col("Number non-green")*100/pl.col("Total number")).alias("% non-green"),
    (pl.col("Number green")*100/pl.col("Total number")).alias("% green"),
    (pl.col("Number with no SOC")*100/pl.col("Total number")).alias("% no soc")
)
grouped_df.write_csv("grouped_green_not_green_props.csv")

In [188]:
grouped_df = (welsh_combined_all_data.group_by("year", maintain_order=True)
     .agg(
         (pl.col("GREEN/NOT GREEN")=="Non-green").sum().alias("Number non-green"),
         (pl.col("GREEN/NOT GREEN")=="Green").sum().alias("Number green"),
         (pl.col("GREEN/NOT GREEN").is_null()).sum().alias("Number with no SOC"),
         pl.col("GREEN/NOT GREEN").len().alias("Total number"),
     ))

grouped_df = grouped_df.with_columns(
    (pl.col("Number non-green")*100/pl.col("Total number")).alias("% non-green"),
    (pl.col("Number green")*100/pl.col("Total number")).alias("% green"),
    (pl.col("Number with no SOC")*100/pl.col("Total number")).alias("% no soc")
)
grouped_df.filter(pl.col("year")!=2020).write_csv("grouped_green_not_green_props_year.csv")

In [262]:
grouped_df_by_country = (combined_all_data.group_by("country", maintain_order=True)
     .agg(
         (pl.col("GREEN/NOT GREEN")=="Non-green").sum().alias("Number non-green"),
         (pl.col("GREEN/NOT GREEN")=="Green").sum().alias("Number green"),
         (pl.col("GREEN/NOT GREEN").is_null()).sum().alias("Number with no SOC"),
         pl.col("GREEN/NOT GREEN").len().alias("Total number"),
     ))

grouped_df_by_country = grouped_df_by_country.with_columns(
    (pl.col("Number non-green")*100/pl.col("Total number")).alias("% non-green"),
    (pl.col("Number green")*100/pl.col("Total number")).alias("% green"),
    (pl.col("Number with no SOC")*100/pl.col("Total number")).alias("% no soc")
)
grouped_df_by_country.filter(~pl.col("country").is_null()
                            ).write_csv("grouped_green_not_green_props_country.csv")

In [263]:
grouped_df_by_itl3 = (welsh_combined_all_data.group_by("itl_3_name", maintain_order=True)
     .agg(
         (pl.col("GREEN/NOT GREEN")=="Non-green").sum().alias("Number non-green"),
         (pl.col("GREEN/NOT GREEN")=="Green").sum().alias("Number green"),
         (pl.col("GREEN/NOT GREEN").is_null()).sum().alias("Number with no SOC"),
         pl.col("GREEN/NOT GREEN").len().alias("Total number"),
     ))

grouped_df_by_itl3 = grouped_df_by_itl3.with_columns(
    (pl.col("Number non-green")*100/pl.col("Total number")).alias("% non-green"),
    (pl.col("Number green")*100/pl.col("Total number")).alias("% green"),
    (pl.col("Number with no SOC")*100/pl.col("Total number")).alias("% no soc")
)
grouped_df_by_itl3.filter(~pl.col("itl_3_name").is_null()
                            ).write_csv("grouped_green_not_green_props_itl3.csv")

## Geography
- Within Wales, what is the geography of green job offers? Is there a South / North divide?

In [208]:
region_grouped_df_pl = (welsh_combined_all_data.group_by("itl_3_code", maintain_order=True)
     .agg(
         pl.col("PROP_GREEN").mean().alias("average_prop_green_skills"),
         pl.col('INDUSTRY GHG PER UNIT EMISSIONS').mean().alias("average_ghg"),
         pl.col('GREEN TIMESHARE').mean().alias("average_green_timeshare"),
         pl.col('job_id').n_unique().alias("number_job_ids"),
     ))

region_grouped_df = region_grouped_df_pl.to_pandas()

In [200]:
import geopandas as gpd
import pandas as pd

In [192]:
from dap_prinz_green_jobs.utils.chloropleth_utils import get_nuts2polygons_dict, get_nuts1polygons_dict, get_nuts3polygons_dict

In [193]:
nuts1polygons_dict = get_nuts1polygons_dict()
itl1polygons_dict = {k.replace("UK","TL"):v for k, v in nuts1polygons_dict.items()}

nuts2polygons_dict = get_nuts2polygons_dict()
itl2polygons_dict = {k.replace("UK","TL"):v for k, v in nuts2polygons_dict.items()}

nuts3polygons_dict = get_nuts3polygons_dict()
itl3polygons_dict = {k.replace("UK","TL"):v for k, v in nuts3polygons_dict.items()}

allpolygons_dict = {**itl1polygons_dict, **itl2polygons_dict, **itl3polygons_dict}

In [201]:

region_grouped_df['geometry_name'] = region_grouped_df["itl_3_code"].map(allpolygons_dict)
region_grouped_df[['geometry', 'itl_name']] = region_grouped_df['geometry_name'].apply(lambda x: pd.Series(x))
region_grouped_df.drop('geometry_name', axis=1, inplace=True)

geo_df = gpd.GeoDataFrame(region_grouped_df)

In [204]:
geo_df.to_file("choropleth_data.geojson", driver="GeoJSON")  

In [212]:
itl_3_name_averages = get_green_averages(welsh_combined_all_data, "itl_3_name")

In [215]:
itl_3_name_averages.write_csv("itl_3_name_averages.csv")

## Green measures, occupations and salaries

- The greenest occupations in Wales are...
- Salaries correlate with greenness as such...

In [219]:
group_by_col = "SOC_2020_name"
welsh_grouped_by_occ = (welsh_combined_all_data.group_by(
    group_by_col, maintain_order=True)
     .agg(
         pl.col("PROP_GREEN").mean().alias("average_prop_green_skills"),
         pl.col('INDUSTRY GHG PER UNIT EMISSIONS').mean().alias("average_ghg"),
         pl.col('GREEN TIMESHARE').mean().alias("average_green_timeshare"),
         pl.col('job_id').n_unique().alias("number_job_ids"),
         pl.col('max_annualised_salary').mean().alias("average_max_annualised_salary"),
     ))

# welsh_grouped_by_occ.unpivot(
#         on=[
#             "average_prop_green_skills",
#             "average_ghg",
#             "average_green_timeshare",
#             "number_job_ids",
#             "average_max_annualised_salary"
#         ],
#         index=group_by_col)

In [240]:
soc_2_onet = dict(zip(welsh_combined_all_data['SOC_2020_name'], welsh_combined_all_data['GREEN/NOT GREEN']))

In [243]:
welsh_grouped_by_occ=welsh_grouped_by_occ.with_columns(pl.col("SOC_2020_name").replace_strict(soc_2_onet, default=None).alias('GREEN/NOT GREEN'))

In [244]:
welsh_grouped_by_occ.filter(
    (~pl.col("SOC_2020_name").is_null()) & (pl.col('number_job_ids')>50)
).write_csv("welsh_grouped_by_occ.csv")